In [33]:
!pip install openmeteo-requests retry-requests requests-cache

In [34]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import matplotlib.pyplot as plt
import os
from datetime import datetime, timedelta

def setup_client():
    """
    Menyiapkan client API dengan fitur Cache dan Retry.
    Tujuannya agar koneksi stabil dan hemat kuota.
    """
    # Buat cache bernama '.cache', data disimpan selama 3600 detik (1 jam)
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)

    # Jika gagal connect, coba lagi (retry) sampai 5x
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)

    # Return objek client yang siap pakai
    return openmeteo_requests.Client(session=retry_session)

In [35]:
# ==========================================
# 2. FETCH DATA PER CHUNK
# ==========================================
def fetch_chunk(client, lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "dew_point_2m",
            "rain",
            "wind_speed_10m",
            "wind_direction_10m",
            "surface_pressure",
            "weather_code"
            ],
        "timezone": "Asia/Jakarta"
    }

    print(f"   ⏳ Mengambil: {start_date} s.d {end_date}...")
    try:
        responses = client.weather_api(url, params=params)
        return process_data(responses[0]) 
    except Exception as e:
        print(f"   ❌ Gagal pada chunk ini: {e}")
        return None

# ==========================================
# 3. PROCESS DATA (UBAHAN: Date Jadi Kolom Biasa)
# ==========================================
def process_data(response):
    hourly = response.Hourly()

    date_range = pd.date_range(
        start = pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end = pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq = pd.Timedelta(seconds=hourly.Interval()),
        inclusive = "left"
    )

    df = pd.DataFrame(data = {
        "date": date_range, # Date tetap jadi kolom biasa
        "temperature": hourly.Variables(0).ValuesAsNumpy(),
        "humidity": hourly.Variables(1).ValuesAsNumpy(),
        "dewpoint":hourly.Variables(2).ValuesAsNumpy(),
        "rain_mm": hourly.Variables(3).ValuesAsNumpy(),
        "wind_speed": hourly.Variables(4).ValuesAsNumpy(),
        "wind_direction": hourly.Variables(5).ValuesAsNumpy(),
        "pressure": hourly.Variables(6).ValuesAsNumpy(),
        "weather_code": hourly.Variables(7).ValuesAsNumpy()
    })

    # Konversi timezone kolom date (Tanpa set_index)
    df['date'] = df['date'].dt.tz_convert('Asia/Jakarta')
    
    return df

# ==========================================
# 4. LOGIKA UTAMA: CHUNKING & MERGING (SAFE MODE)
# ==========================================
def fetch_long_period_data(client, lat, lon, start_str, end_str, folder_tujuan, chunk_years=10):

    if not os.path.exists(folder_tujuan):
        os.makedirs(folder_tujuan)

    start_date = datetime.strptime(start_str, "%Y-%m-%d")
    final_end_date = datetime.strptime(end_str, "%Y-%m-%d")

    all_files = []
    current_start = start_date

    print(f"🚀 Memulai Misi Pengambilan Data...")

    while current_start < final_end_date:
        current_end = current_start.replace(year=current_start.year + chunk_years) - timedelta(days=1)
        if current_end > final_end_date:
            current_end = final_end_date

        s_str = current_start.strftime("%Y-%m-%d")
        e_str = current_end.strftime("%Y-%m-%d")
        
        chunk_filename = os.path.join(folder_tujuan, f"temp_{s_str}_{e_str}.csv")

        if os.path.exists(chunk_filename):
            print(f"⏩ Skip: {s_str} - {e_str} (Sudah ada)")
            all_files.append(chunk_filename)
        else:
            df_chunk = fetch_chunk(client, lat, lon, s_str, e_str)
            if df_chunk is not None:
                # SIMPAN TANPA INDEX (index=False)
                # Biar kolom 'date' tersimpan rapi sebagai kolom pertama
                df_chunk.to_csv(chunk_filename, index=False)
                print(f"   ✅ Tersimpan: {chunk_filename}")
                all_files.append(chunk_filename)
                
                # Jeda sebentar biar API gak ngambek
                time.sleep(2) 
            else:
                print("   ⚠️ Chunk ini dilewati karena error.")

        current_start = current_end + timedelta(days=1)

    print("\n🔗 Menggabungkan semua pecahan data...")

    # Gabungkan
    df_list = []
    for f in all_files:
        # BACA SEBAGAI KOLOM BIASA (Tanpa index_col)
        df = pd.read_csv(f)
        df_list.append(df)

    if df_list:
        df_final = pd.concat(df_list, ignore_index=True)
        
        # Pastikan kolom date dibaca sebagai datetime
        df_final['date'] = pd.to_datetime(df_final['date'])
        
        # Urutkan berdasarkan kolom date
        df_final = df_final.sort_values('date')
        
        # Hapus duplikat
        df_final = df_final.drop_duplicates(subset=['date'], keep='first')

        print(f"🎉 SUKSES BESAR! Total Data: {len(df_final)} baris.")
        print(f"   Contoh Data (5 Teratas):")
        print(df_final.head()) # Cek apakah tanggalnya muncul
        print(f"   Mulai: {df_final['date'].min()}")
        print(f"   Akhir: {df_final['date'].max()}")
        
        return df_final
    else:
        return None

In [36]:
import time
from datetime import date, timedelta
import os
# Pastikan library lain yang dibutuhkan (openmeteo_requests, pandas, retry, dll) sudah terimport di bagian atas filemu

# ==========================================
# FUNGSI TAMBAHAN: RETRY LOGIC (ANTI GAGAL)
# ==========================================
def fetch_chunk_safe(func_fetch, *args, max_retries=5, **kwargs):
    """
    Mencoba menjalankan fungsi fetch. Jika gagal (kena limit API),
    akan menunggu dan mencoba lagi secara otomatis.
    """
    wait_seconds = 60 # Tunggu 1 menit jika gagal (sesuai aturan Open Meteo)

    for attempt in range(1, max_retries + 1):
        try:
            return func_fetch(*args, **kwargs)
        except Exception as e:
            print(f"⚠️ Percobaan ke-{attempt} gagal: {e}")
            if attempt < max_retries:
                print(f"⏳ Menunggu {wait_seconds} detik sebelum mencoba lagi...")
                time.sleep(wait_seconds)
            else:
                print("❌ Gagal total setelah beberapa kali percobaan.")
                raise e # Lempar error jika sudah menyerah

# ==========================================
# EKSEKUSI UTAMA
# ==========================================
if __name__ == "__main__":
    # 1. Konfigurasi Lokasi
    # Perhatikan: tuple koma di akhir LAT dihapus agar aman jadi float biasa
    LAT = -7.736436737566032
    LON = 109.6460550796716

    # 2. Konfigurasi Waktu (OTOMATIS)
    MULAI = "1950-01-01"

    # Hitung tanggal kemarin (H-1) secara otomatis
    kemarin = date.today() - timedelta(days=1)
    AKHIR = kemarin.strftime("%Y-%m-%d")

    print(f"📅 Jadwal Run Otomatis: Mengambil data dari {MULAI} sampai {AKHIR}")

    # 3. Konfigurasi File
    FOLDER = "open_meteo_climate"
    FILE_FINAL = "kebumen_75tahun_lengkap.csv"

    # Setup Client (Pastikan fungsi setup_client() ada di kodemu sebelumnya)
    client = setup_client()

    # 4. Jalankan Fetching dengan Pengaman (Safe Mode)
    # Kita bungkus proses fetching utama dalam try-except besar atau gunakan logika retry di level chunk
    # Namun, karena fetch_long_period_data sudah melakukan looping internal,
    # kita modifikasi sedikit cara panggilannya atau pastikan fungsi fetch_long_period_data
    # memiliki jeda (sleep) antar request.

    try:
        # Jalankan fungsi utamamu
        # Tips: Open-Meteo membatasi request per menit.
        # Pastikan di dalam fungsi `fetch_long_period_data` ada time.sleep(5) antar chunk.
        df_lengkap = fetch_long_period_data(client, LAT, LON, MULAI, AKHIR, FOLDER, chunk_years=5)

        if df_lengkap is not None:
            # Simpan Hasil Akhir
            if not os.path.exists(FOLDER):
                os.makedirs(FOLDER)

            path_final = os.path.join(FOLDER, FILE_FINAL)
            df_lengkap.to_csv(path_final, index=False) # index=False biar rapi
            print(f"✅ SUKSES! File Gabungan Tersimpan: {path_final}")

            # Hapus file temp (Opsional)
            # import glob
            # for f in glob.glob(f"{FOLDER}/temp_*.csv"):
            #     os.remove(f)

    except Exception as e:
        print(f"❌ Terjadi kesalahan fatal pada script utama: {e}")

📅 Jadwal Run Otomatis: Mengambil data dari 1950-01-01 sampai 2025-12-06
🚀 Memulai Misi Pengambilan Data...
⏩ Skip: 1950-01-01 - 1954-12-31 (Sudah ada)
⏩ Skip: 1955-01-01 - 1959-12-31 (Sudah ada)
⏩ Skip: 1960-01-01 - 1964-12-31 (Sudah ada)
⏩ Skip: 1965-01-01 - 1969-12-31 (Sudah ada)
⏩ Skip: 1970-01-01 - 1974-12-31 (Sudah ada)
⏩ Skip: 1975-01-01 - 1979-12-31 (Sudah ada)
⏩ Skip: 1980-01-01 - 1984-12-31 (Sudah ada)
⏩ Skip: 1985-01-01 - 1989-12-31 (Sudah ada)
⏩ Skip: 1990-01-01 - 1994-12-31 (Sudah ada)
⏩ Skip: 1995-01-01 - 1999-12-31 (Sudah ada)
⏩ Skip: 2000-01-01 - 2004-12-31 (Sudah ada)
⏩ Skip: 2005-01-01 - 2009-12-31 (Sudah ada)
⏩ Skip: 2010-01-01 - 2014-12-31 (Sudah ada)
⏩ Skip: 2015-01-01 - 2019-12-31 (Sudah ada)
⏩ Skip: 2020-01-01 - 2024-12-31 (Sudah ada)
⏩ Skip: 2025-01-01 - 2025-12-06 (Sudah ada)

🔗 Menggabungkan semua pecahan data...


/tmp/ipykernel_47/2732877289.py:120: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df_final['date'] = pd.to_datetime(df_final['date'])


🎉 SUKSES BESAR! Total Data: 665616 baris.
   Contoh Data (5 Teratas):
                        date  temperature  humidity   dewpoint  rain_mm  \
0  1950-01-01 01:00:00+08:00    23.904501  97.03363  23.404501      0.0   
1  1950-01-01 02:00:00+08:00    23.904501  97.03363  23.404501      0.0   
2  1950-01-01 03:00:00+08:00    24.054500  95.58440  23.304500      0.0   
3  1950-01-01 04:00:00+08:00    23.404501  96.72919  22.854500      0.0   
4  1950-01-01 05:00:00+08:00    23.654501  95.86120  22.954500      0.0   

   wind_speed  wind_direction    pressure  weather_code  
0    3.240000       360.00000  1002.40784           3.0  
1    3.960000       360.00000  1001.60956           1.0  
2    4.802999       347.00537  1001.41080           2.0  
3    4.735060       351.25390  1001.30646           3.0  
4    5.411986       356.18600  1001.40810           3.0  
   Mulai: 1950-01-01 01:00:00+08:00
   Akhir: 2025-12-06 23:00:00+07:00
✅ SUKSES! File Gabungan Tersimpan: open_meteo_climate/kebum

In [37]:
data = pd.read_csv("/kaggle/working/open_meteo_climate/kebumen_75tahun_lengkap.csv")
data.tail(10)

,date,temperature,humidity,dewpoint,rain_mm,wind_speed,wind_direction,pressure,weather_code
665606,2025-12-06 14:00:00+07:00,26.338000,85.643456,23.737999,0.9,3.438895,312.878900,1006.91560,53.0
665607,2025-12-06 15:00:00+07:00,26.737999,85.940410,24.188000,0.3,0.540000,180.000000,1006.32007,51.0
665608,2025-12-06 16:00:00+07:00,26.938000,84.427315,24.088000,0.2,3.244996,146.309900,1006.42114,51.0
665609,2025-12-06 17:00:00+07:00,26.138000,83.076805,23.038000,0.2,5.081613,157.067870,1006.21594,51.0
665610,2025-12-06 18:00:00+07:00,25.538000,89.797640,23.737999,0.2,2.545584,81.869990,1006.61066,51.0
665611,2025-12-06 19:00:00+07:00,25.538000,92.813650,24.288000,0.0,2.346913,32.471172,1007.40890,3.0
665612,2025-12-06 20:00:00+07:00,25.288000,94.769290,24.388000,0.1,2.811690,50.194473,1008.20526,51.0
665613,2025-12-06 21:00:00+07:00,25.138000,95.906000,24.438000,0.0,3.864764,27.758451,1008.70325,3.0
665614,2025-12-06 22:00:00+07:00,24.688000,95.031770,23.838000,0.2,0.917824,11.309895,1008.89930,51.0
665615,2025-12-06 23:00:00+07:00,24.688000,93.330620,23.538000,0.3,1.451206,209.744800,1008.79950,51.0


In [38]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 665616 entries, 0 to 665615
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   date            665616 non-null  object 
 1   temperature     665616 non-null  float64
 2   humidity        665616 non-null  float64
 3   dewpoint        665616 non-null  float64
 4   rain_mm         665616 non-null  float64
 5   wind_speed      665616 non-null  float64
 6   wind_direction  665616 non-null  float64
 7   pressure        665616 non-null  float64
 8   weather_code    665616 non-null  float64
dtypes: float64(8), object(1)
memory usage: 45.7+ MB
